In [2]:
# import knihoven
import pandas as pd
import numpy as np
from uncertainties import ufloat as uf
from pathlib import Path

# path config
downloads_dir = Path.home() / "Downloads"
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [21]:
# definice funkci

# nejistoty typu B pro digitalni pristroje
def unc_B_digital(reading, percent_reading, digits, resolution):    # vse musi byt ve spravnych jednotkach; funguje pouze pro jedny podminky zaroven
    """
    Vypocet nejistoty typu B pro digitalni mereni (ponechame krajni)

    reading : float nebo array
        namerena hodnota
    percent_reading : float
        % of reading
    digits : float
        pocet digitu
    resolution : float
        nejmensi dilek
    """

    max_error = abs(reading) * (percent_reading/100) + digits * resolution  # max_error -- krajni nejistota
    u_B = max_error

    return u_B

# nejistota typu A 
def unc_A(data):
    """
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    """
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

In [59]:
# nacteni google tabulek
sheet_id = '1X6DslFJgiO2eAeN-leIRMBTIs06rjy_FhxOTD-FuwGI'
gid = '0'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_cocky = pd.read_csv(url)
df_cocky.dropna()

,spojka,stinitko,zdroj,sipka,delta,predmet,rozptylka,obraz,pocatek
0,29.4,75.0,5.0,8.1,47.6,69.0,56.6,81,4.1


In [68]:
# cocky
spojka, stinitko, zdroj, sipka, delta, predmet, rozptylka, obraz, pocatek = (df_cocky[c].dropna().to_numpy()*1e-2 for c in df_cocky.columns)
pocatek = pocatek[0]
zdroj = zdroj[0]
veliciny = [spojka, stinitko, delta, predmet, rozptylka, obraz]
veliciny = [v - pocatek for v in veliciny]
[spojka, stinitko, delta, predmet, rozptylka, obraz] = veliciny
predmet = predmet[0]
spojka = -spojka
f_prima = (spojka*stinitko)/(spojka-stinitko)
#print(f_prima)

zvetseni = sipka/zdroj
f_zvetseni = (spojka*zvetseni)/(1-zvetseni)
#print(f_zvetseni)

delta = delta + spojka
f_bessel = (stinitko**2 - delta**2)/(4*stinitko)
#print(f_bessel)

predmet_roz = rozptylka - predmet
stinitko_roz = obraz - predmet
f_rozptylka = (predmet_roz*stinitko_roz)/(predmet_roz-stinitko_roz)
print(f_rozptylka)

[0.06098361 0.06872727 0.07438127 0.08       0.08603989 0.09161804
 0.09742574 0.10242424 0.05412844 0.04521739 0.03464789]


In [18]:
# nacteni google tabulek
sheet_id = '1X6DslFJgiO2eAeN-leIRMBTIs06rjy_FhxOTD-FuwGI'
gid = '708172543'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_sfero = pd.read_csv(url)
df_sfero.dropna()

,h_s1,z_s,h_s2,h_r1,z_r,h_r2
0,0.004,34.92,1.834,0.506,37.17,0.508
1,0.005,34.98,1.835,0.505,37.15,0.508
2,0.004,34.99,1.835,0.506,37.17,0.509
3,0.004,34.97,1.836,0.506,37.18,0.508
4,0.004,35.00,1.835,0.505,37.17,0.509


In [ ]:
# sferometr
h_s1, z_s, h_s2, h_r1, z_r, h_r2= (df_sfero[c].dropna().to_numpy()*1e-3 for c in df_sfero.columns)
h_s1 = uf(np.mean(h_s1), unc_A(h_s1))
z_s = uf(np.mean(z_s), unc_A(z_s))
h_s2 = uf(np.mean(h_s2), unc_A(h_s2))

h_r1 = uf(-np.mean(h_r1), unc_A(h_r1))
z_r = uf(np.mean(z_r), unc_A(z_r))
h_r2 = uf(-np.mean(h_r2), unc_A(h_r2))


r_s1 = (z_s**2 + h_s1**2)/(2*h_s1)
r_s2 = (z_s**2 + h_s2**2)/(2*h_s2)

r_r1 = (z_r**2 + h_r1**2)/(2*h_r1)
r_r2 = (z_r**2 + h_r2**2)/(2*h_r2)



In [ ]:
n_p = (1+ 1/f_P)/(1/r_s1 - 1/(r_s2))